# Initial EDA — User Coverage Across Tables

Explore how many unique `msno` (user IDs) exist in each of the 4 Delta tables
and identify the set of users that appear in **all** tables.

In [1]:
import os

from databricks.connect import DatabricksSession

os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "llmops-course")
spark = DatabricksSession.builder.profile("llmops-course").getOrCreate()

In [2]:
CATALOG = "mlops_dev"
SCHEMA = "chenheju"

members = spark.table(f"{CATALOG}.{SCHEMA}.members")
train = spark.table(f"{CATALOG}.{SCHEMA}.train")
transactions = spark.table(f"{CATALOG}.{SCHEMA}.transactions")
user_logs = spark.table(f"{CATALOG}.{SCHEMA}.user_logs")

## Unique `msno` per Table

In [3]:
tables = {
    "members": members,
    "train": train,
    "transactions": transactions,
    "user_logs": user_logs,
}

unique_counts = {}
for name, df in tables.items():
    total = df.count()
    unique = df.select("msno").distinct().count()
    unique_counts[name] = unique
    print(f"{name:15s}  total rows: {total:>12,}  |  unique msno: {unique:>10,}")

members          total rows:    6,769,473  |  unique msno:  6,769,473
train            total rows:      970,960  |  unique msno:    970,960
transactions     total rows:    1,431,009  |  unique msno:  1,197,050
user_logs        total rows:   18,396,362  |  unique msno:  1,103,894


## Users Present in ALL 4 Tables

In [4]:
members_ids = members.select("msno").distinct()
train_ids = train.select("msno").distinct()
transactions_ids = transactions.select("msno").distinct()
user_logs_ids = user_logs.select("msno").distinct()

common_ids = (
    members_ids.intersect(train_ids).intersect(transactions_ids).intersect(user_logs_ids)
)

common_count = common_ids.count()
print(f"Users in ALL 4 tables: {common_count:,}")

Users in ALL 4 tables: 725,722


## Coverage Breakdown

In [5]:
print(f"{'Table':<15s}  {'Unique msno':>12s}  {'In common set':>14s}  {'Coverage':>8s}")
print("-" * 60)
for name, _df in tables.items():
    unique = unique_counts[name]
    pct = common_count / unique * 100 if unique else 0
    print(f"{name:<15s}  {unique:>12,}  {common_count:>14,}  {pct:>7.1f}%")

print(f"\nKeeping only common users would retain {common_count:,} unique msno values.")

Table             Unique msno   In common set  Coverage
------------------------------------------------------------
members             6,769,473         725,722     10.7%
train                 970,960         725,722     74.7%
transactions        1,197,050         725,722     60.6%
user_logs           1,103,894         725,722     65.7%

Keeping only common users would retain 725,722 unique msno values.


## Pairwise Overlap

In [6]:
from itertools import combinations

id_sets = {
    "members": members_ids,
    "train": train_ids,
    "transactions": transactions_ids,
    "user_logs": user_logs_ids,
}

print(f"{'Pair':<35s}  {'Overlap':>10s}")
print("-" * 50)
for a, b in combinations(id_sets, 2):
    overlap = id_sets[a].intersect(id_sets[b]).count()
    print(f"{a + ' ∩ ' + b:<35s}  {overlap:>10,}")

Pair                                    Overlap
--------------------------------------------------
members ∩ train                         860,967
members ∩ transactions                1,077,434
members ∩ user_logs                   1,103,854
train ∩ transactions                    933,578
train ∩ user_logs                       754,551
transactions ∩ user_logs                967,044
